---
title: FASTQ Format Validation
subtitle: SAM/BAM file format checks
date: "9999-12-31"
edit_url: null
---
**Linked-read type**: PLACEHOLDER

In [ ]:
from harpy.report.components import ITable, JSFunction, StatsBox
import pandas as pd


In [ ]:
infile = "validate.fastq.tsv"


This report reflects the FASTQ files Harpy processed to identify obvious
formatting issues that may require your attention.

In [ ]:
data = pd.read_table(infile, sep=r'\s+')
attention_df = (data
    .assign(allmissing = lambda x: x['reads'] == x['noBX'])
    .iloc[:, [3, 4, 6]]
    .apply(lambda row: row[['badBX', 'badSamSpec', 'allmissing']].sum(), axis=1)
)
attention = (attention_df > 0).sum()
bxnotlast = (data['bxNotLast'] > 0).sum()

(
    StatsBox()
    .add(len(data.index), "Files")
    .conditional(attention, "Needs Review", 1, lower_bad=False)
    .conditional(bxnotlast, "BX Not Last", 1, lower_bad=False)
    .render()
)


## Metrics
The `harpy validate fastq` command created a `validate.fastq.tsv` file in the specified
output directory that summarizes the results that are included in this report. This file
contains a tab-delimited table with the columns: `file`, `reads`, `noBX`, `badBX`, and `badSamSpec`.

:::{note} Validation logic
:open: false
Severity given by: ⬜ = mild | 🔶 = moderate  | 🛑 = serious
| column        | severity | pass condition                                                        | fail condition 🟨                                       |
|:--------------|:---|:----------------------------------------------------------------------------|:-----------------------------------------------------------|
| **badBX**    | 🛑 | **all** reads with barcodes are properly formatted for their chemistry | **any** barcodes are incorrectly formatted barcodes their chemistry   |
| **badSamSpec**  | 🛑 | **all** reads have proper `TAG:TYPE:VALUE` comments                         | **any** reads have incorrectly formatted comments          |
| **bxNotLast** | ⬜/🔶 | **all** reads have `BX:Z` as final comment                                  | **at least 1 read** doesn’t have BX:Z tag as final comment [^1] |
| **noBX**  | ⬜/🔶 | **any** barcodes present                                                 | **all** reads lack barcodes                                |

:::
[^1]:
    Things regarding `BX:Z` tags are ignored in non-haplotagging chemistries

In [ ]:
tbl = ITable(data, "validate.fastq.csv", fixedcols=1)
for i in range(1, len(tbl.col_defs)):
    tbl.col_defs[i]['cellStyle'] = JSFunction("""
        params => {
            if (params.value > '0') {
                return {color: 'black', backgroundColor: '#f6ab3c'};
            }
            return null;
        }
    """)
tbl.render()

## Supporting info
:::{dropdown} Understanding the columns
:open:

**file:** The name of the FASTQ file.

**reads**: The total number of reads in the file.

**noBX**: The number of reads that do not have a barcode present.

**badBX**: The barcode does is not in the proper chemistry-specific format.

**badSamSpec**: The comments in the read header after the read ID do not conform to the `TAG:TYPE:VALUE` [SAM specification](https://samtools.github.io/hts-specs/SAMv1.pdf).

**bxNotLast**: The `BX:Z:` tag in the FASTQ header is not the last comment. 
Only relevant for LEVIATHAN variant calling and can be ignored if not intending to call
structural variants with LEVIATHAN,otherwise the `BX:Z` tag must be the last comment in a read
header. `harpy align` will automatically move the `BX:Z` tag to the end of the alignment record.
:::

:::{dropdown} What is "valid"?
A "valid" linked-read fastq file will vary depending on the linked-read chemistry, but their
differences are exclusively in the read headers[^2].
Note that the `invalidated` definitions provided below refer to how each chemistry
defines an "invalid barcode", meaning it adheres to the chemistry's format, but specifies
it is not reliably linked to any other molecule. The formats are:

**haplotagging**: `@SEQID:VAL:VAL/1  BX:Z:AXXCXXBXXDXX` 
- barcode: `AXXCXXBXXDXX` where `X` is a 0-9 integer
- invalidated: if any position is `00` (e.g. `A91C00B42D57`)

**stlfr**: `@SEQID:VAL:VAL#X_X_X    1:N:ATGACA`
- barcode: `X_X_X` where  `X` is any integer
- invalidated: if any position is `0` (e.g. `51_0_492`)

**tellseq**: `@SEQID:VAL:VAL:ATCGN    1:N:ATGACA`
- barcode: `ATCG` nucleotide letters
- invalidated: if any nucleotide is `N` (e.g. `ATACANGGAT`)

---

**Example Header**

The read header below contains the read ID `@A00814...`,
and the `BX:Z` barcode tag (haplotagging format), which adheres to the [SAM specification](https://samtools.github.io/hts-specs/SAMv1.pdf)
of `TAG:TYPE:VALUE`. Any comments, if present, must be **TAB separated**.
```
@A00814:267:HTMH3DRXX:2:1101:4580:1000/1	BX:Z:A02C01B11D46
```
:::

[^2]:
    10X format is the oddball here, since it does not store the barcode in the read headers and instead stores it in the first 16
    base pairs of read 1 (the forward read).